In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

'/projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene'

In [2]:
import sys, os

def add_path(p: str):
    if p not in sys.path:
        sys.path.append(p)

add_path(os.environ['LP_ROOT'])
import project

add_path(os.environ['PROJECT'] + '/' + 'param_search')
import param_search as ps

ps.set_backend('sge')
ps.set_verbose(False)

# Gather examples

In [3]:
from pathlib import Path
#data_root = Path(os.environ['LP_ROOT'] / 'data' / 'COPDGene'
data_root = Path(os.environ['PRIVATE']) / 'data' / 'COPDGene'
data_root.is_dir()

True

In [4]:
import pandas as pd
subject_file = data_root / 'sample1000_2025-07-22.csv'
subject_list = list(pd.read_csv(subject_file, sep='\t').sid)
len(subject_list)

1000

# Setup experiment

In [5]:
base_dir = '2026-09-15_preprocess'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=32G
#$ -l h_rt=3:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.name={data_name} \\
    --set dataset.root={data_root} \\
    --set dataset.examples.subjects=[{subject}] \\
    --set dataset.examples.variant={variant} \\
    --set dataset.examples.pipeline_tags.material_properties={mat_tag} \\
    --set dataset.examples.pipeline_tags.forward_simulation={fwd_tag} \\
    --set preprocessing.forward_simulation.pde_solver.material_type={mat_type}

'''
name_format = 'job_{params_hash}'

grid = ps.param_grid(
    config='2026-09-15_config.yaml',
    data_name='COPDGene',
    data_root=str(data_root),
    subject=subject_list,
    mat_tag='bela',
    fwd_tag='linear',
    mat_type='linear',
    variant='2026-08-08'
)

ps.param_grid( # TODO
    config='2026-08-08_config.yaml',
    data_name='COPDGene',
    data_root=str(data_root),
    subject=subject_list,
    mat_tag='bela',
    fwd_tag='stvk',
    mat_type='stvk',
    variant='2026-08-08'
)

len(grid)

1000

In [6]:
%autoreload
try:
    jobs = ps.setup(base_dir, template, name_format, grid, overwrite=False)
except OSError:
    jobs = ps.load(base_dir)

jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params_json,params_hash,params.config,params.data_name,params.data_root,params.subject,params.mat_tag,params.fwd_tag,params.mat_type,params.variant
0,job_ead9ab1e5bc03c66,SUBMITTED,1,7207880,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",ead9ab1e5bc03c66,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16514P,bela,linear,linear,2026-08-08
1,job_96ef0272ef81ee7e,SUBMITTED,1,7207881,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",96ef0272ef81ee7e,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20748Q,bela,linear,linear,2026-08-08
2,job_4d5d759871906077,SUBMITTED,1,7207882,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",4d5d759871906077,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,11007Z,bela,linear,linear,2026-08-08
3,job_62c1bdeb21500ce0,SUBMITTED,1,7207883,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",62c1bdeb21500ce0,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,14771Z,bela,linear,linear,2026-08-08
4,job_1a856f1cecae8f51,SUBMITTED,1,7207884,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",1a856f1cecae8f51,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,13651K,bela,linear,linear,2026-08-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job_22f2adacb02a421c,SUBMITTED,1,7208895,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",22f2adacb02a421c,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20519B,bela,linear,linear,2026-08-08
996,job_b1e67f498dc392b6,SUBMITTED,1,7208896,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",b1e67f498dc392b6,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,12294H,bela,linear,linear,2026-08-08
997,job_2f617f39d23060c8,SUBMITTED,1,7208897,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",2f617f39d23060c8,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,23123R,bela,linear,linear,2026-08-08
998,job_bef061b64dfcfc4e,SUBMITTED,1,7208898,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",bef061b64dfcfc4e,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16546C,bela,linear,linear,2026-08-08


In [19]:
jobs = ps.submit(jobs)

In [7]:
%autoreload
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs)
jobs = ps.collect(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.mat_type,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,job_ead9ab1e5bc03c66,SUBMITTED,1,7207880,None,None,INFO: /restricted/projectnb/batmanlab/mragoza/...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
1,job_96ef0272ef81ee7e,SUBMITTED,1,7207881,None,None,INFO: /restricted/projectnb/batmanlab/mragoza/...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
2,job_4d5d759871906077,SUBMITTED,1,7207882,None,None,INFO: /restricted/projectnb/batmanlab/mragoza/...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
3,job_62c1bdeb21500ce0,SUBMITTED,1,7207883,None,None,INFO: /restricted/projectnb/batmanlab/mragoza/...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
4,job_1a856f1cecae8f51,SUBMITTED,1,7207884,None,None,INFO: /restricted/projectnb/batmanlab/mragoza/...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job_22f2adacb02a421c,SUBMITTED,1,7208895,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
996,job_b1e67f498dc392b6,SUBMITTED,1,7208896,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
997,job_2f617f39d23060c8,SUBMITTED,1,7208897,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
998,job_bef061b64dfcfc4e,SUBMITTED,1,7208898,None,None,Loading /restricted/projectnb/batmanlab/mragoz...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>


In [8]:
jobs.groupby('job_state').count()

,job_name,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,script_path,...,params.mat_type,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
job_state,,,,,,,,,,,,,,,,,,,,,
SUBMITTED,1000,1000,1000,0,0,1000,1000,1000,1000,1000,...,1000,1000,0,0,0,0,0,1000,0,0


In [26]:
def classify_error(stderr):
    if pd.isnull(stderr):
        return 'No stderr file'
    elif not str(stderr).strip():
        return 'No error message'
    elif 'CUDA out of memory' in stderr or ('Failed to allocate' in stderr and "bytes on device 'cuda" in stderr):
        return 'GPU out of memory'
    elif 'Non-positive density' in stderr:
        return 'Non-positive density'
    elif 'construct initial points' in stderr or 'Exuding' in stderr or ('Moving #' in stderr and 'addr:' in stderr):
        return 'pygalmesh output'
    return 'Uncategorized error'

jobs['error_type'] = jobs.stderr.map(classify_error)
jobs.groupby('error_type')[['job_name']].count()

,job_name
error_type,
GPU out of memory,7
No error message,979
pygalmesh output,14


In [20]:
jobs[jobs.error == 'Uncategorized error'].iloc[0].stderr

'construct initial points (nb_points: 12)\n'

In [25]:
jobs.loc[:, 'job_id'] = pd.NA

In [28]:
%autoreload
jobs = ps.submit(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params_json,params_hash,params.config,params.data_name,params.data_root,params.subject,params.mat_tag,params.fwd_tag,params.mat_type,params.variant
0,jobead9ab1e5bc03c66,SUBMITTED,1,7159234,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",ead9ab1e5bc03c66,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16514P,bela,linear,linear,2026-08-08
1,job96ef0272ef81ee7e,SUBMITTED,1,7159235,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",96ef0272ef81ee7e,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20748Q,bela,linear,linear,2026-08-08
2,job4d5d759871906077,SUBMITTED,1,7159236,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",4d5d759871906077,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,11007Z,bela,linear,linear,2026-08-08
3,job62c1bdeb21500ce0,SUBMITTED,1,7159237,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",62c1bdeb21500ce0,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,14771Z,bela,linear,linear,2026-08-08
4,job1a856f1cecae8f51,SUBMITTED,1,7159238,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",1a856f1cecae8f51,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,13651K,bela,linear,linear,2026-08-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job22f2adacb02a421c,SUBMITTED,1,7160229,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",22f2adacb02a421c,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20519B,bela,linear,linear,2026-08-08
996,jobb1e67f498dc392b6,SUBMITTED,1,7160230,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",b1e67f498dc392b6,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,12294H,bela,linear,linear,2026-08-08
997,job2f617f39d23060c8,SUBMITTED,1,7160231,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",2f617f39d23060c8,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,23123R,bela,linear,linear,2026-08-08
998,jobbef061b64dfcfc4e,SUBMITTED,1,7160232,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",bef061b64dfcfc4e,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16546C,bela,linear,linear,2026-08-08
